In [ ]:
"""
PASSO 1 — Download dos dados OHLCV da B3 via Yahoo Finance.

Para cada ação no arquivo .NewB3_pruned.csv, baixa dados de 01/01/2016 a 30/06/2020,
ajusta preços usando Adjusted Close (equivalente ao adjustOHLC do R),
e salva como {Code}_New.csv com colunas: Date, Open, High, Low, Close, Volume.

Uso
"""

from pathlib import Path
import pandas as pd
import numpy as np
import time

# ===================== CONFIGURAÇÃO =====================
BASE_DIR = Path(r"C:\Users\Rodrigo\Desktop\Replicando para B3_2\B3ICS")                        
SEC_NAMES = BASE_DIR / ".NewB3_pruned.csv"   # arquivo com Code, industry, Sample
START_DATE = "2016-01-01"
END_DATE   = "2020-06-30"
SLEEP_BETWEEN = 0.25  # segundos entre downloads (evita throttling do Yahoo)
# ========================================================

try:
    import yfinance as yf
except ImportError:
    raise ImportError("Instale yfinance: pip install yfinance")


def read_codes(path: Path) -> pd.DataFrame:
    """Lê o arquivo .NewB3_pruned.csv e retorna DataFrame com Code, industry, Sample."""
    df = pd.read_csv(path, dtype=str, encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]
    df["Code"] = df["Code"].str.strip().str.upper()
    return df


def adjust_ohlcv(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    Ajusta OHLC usando Adj Close (equivalente a quantmod::adjustOHLC no R).
    Retorna DataFrame com colunas [Open, High, Low, Close, Volume].
    """
    if "Adj Close" not in raw_df.columns or "Close" not in raw_df.columns:
        return None

    factor = raw_df["Adj Close"] / raw_df["Close"]
    factor = factor.replace([np.inf, -np.inf], np.nan).ffill().fillna(1.0)

    out = pd.DataFrame(index=raw_df.index)
    out["Open"]   = raw_df["Open"] * factor
    out["High"]   = raw_df["High"] * factor
    out["Low"]    = raw_df["Low"]  * factor
    out["Close"]  = raw_df["Adj Close"]
    out["Volume"] = raw_df["Volume"]
    return out.dropna()


def download_all(sec_names_path: Path, base_dir: Path):
    """Baixa dados para todos os tickers listados no CSV."""
    codes_df = read_codes(sec_names_path)
    codes = codes_df["Code"].tolist()

    print(f"Total de tickers: {len(codes)}")
    print(f"Período: {START_DATE} → {END_DATE}\n")

    report = []
    for i, code in enumerate(codes, 1):
        yahoo_symbol = f"{code}.SA"
        out_file = base_dir / f"{code}_New.csv"

        # Progresso
        print(f"[{i}/{len(codes)}] {code} ... ", end="", flush=True)

        # Se já existe, pula
        if out_file.exists():
            print("já existe, pulando.")
            report.append({"Code": code, "status": "skipped", "rows": 0})
            continue

        try:
            raw = yf.download(
                yahoo_symbol,
                start=START_DATE,
                end=END_DATE,
                progress=False,
                auto_adjust=False,
            )

            if raw is None or raw.empty:
                print("sem dados.")
                report.append({"Code": code, "status": "empty", "rows": 0})
                time.sleep(SLEEP_BETWEEN)
                continue

            # Flatten multi-level columns se necessário (yfinance >= 0.2.x)
            if isinstance(raw.columns, pd.MultiIndex):
                raw.columns = raw.columns.get_level_values(0)

            adj = adjust_ohlcv(raw)
            if adj is None or adj.empty:
                print("falha no ajuste.")
                report.append({"Code": code, "status": "adj_fail", "rows": 0})
                time.sleep(SLEEP_BETWEEN)
                continue

            # Salva: Date como coluna (não index)
            adj_save = adj.reset_index()
            adj_save.rename(columns={adj_save.columns[0]: "Date"}, inplace=True)
            adj_save["Date"] = pd.to_datetime(adj_save["Date"]).dt.strftime("%Y-%m-%d")
            adj_save.to_csv(out_file, index=False, encoding="utf-8-sig")

            print(f"OK ({len(adj)} linhas)")
            report.append({"Code": code, "status": "ok", "rows": len(adj)})

        except Exception as e:
            print(f"ERRO: {e}")
            report.append({"Code": code, "status": f"error: {e}", "rows": 0})

        time.sleep(SLEEP_BETWEEN)

    # Salva relatório
    report_df = pd.DataFrame(report)
    report_file = base_dir / "download_report.csv"
    report_df.to_csv(report_file, index=False, encoding="utf-8-sig")

    # Resumo
    n_ok = (report_df["status"] == "ok").sum()
    n_skip = (report_df["status"] == "skipped").sum()
    n_fail = len(report_df) - n_ok - n_skip
    print(f"\n{'='*50}")
    print(f"Concluído: {n_ok} baixados, {n_skip} já existiam, {n_fail} falharam.")
    print(f"Relatório salvo em: {report_file}")

    return report_df


if __name__ == "__main__":
    download_all(SEC_NAMES, BASE_DIR)

Total de tickers: 303
Período: 2016-01-01 → 2020-06-30

[1/303] CSAN3 ... 

C:\Users\Rodrigo\anaconda3\Lib\site-packages\yfinance\scrapers\history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


OK (1119 linhas)
[2/303] RPMG3 ... 

C:\Users\Rodrigo\anaconda3\Lib\site-packages\yfinance\scrapers\history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


OK (1119 linhas)
[3/303] PETR3 ... 

...

sem dados.
[302/303] VBBR3 ... 

C:\Users\Rodrigo\anaconda3\Lib\site-packages\yfinance\scrapers\history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


OK (567 linhas)
[303/303] OPCT3 ... 

C:\Users\Rodrigo\anaconda3\Lib\site-packages\yfinance\scrapers\history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()

1 Failed download:
['OPCT3.SA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2016-01-01 -> 2020-06-30) (Yahoo error = "Data doesn\'t exist for startDate = 1451613600, endDate = 1593486000")')


sem dados.

Concluído: 260 baixados, 0 já existiam, 43 falharam.
Relatório salvo em: C:\Users\Rodrigo\Desktop\Replicando para B3_2\B3ICS\download_report.csv
